In [7]:
# 02 - CNN Baseline
# Trains a simple from-scratch CNN on data/processed/ (created by 01_data_prep.ipynb)

import sys
sys.path.insert(0, '..')

import os
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

from src.models.cnn_model import SimpleCNN
from src.data.split_dataset import check_dataset

In [2]:
# check device - use GPU if available, otherwise CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


In [3]:
# make sure the split from step 1 actually exists before going further
train_dir = "../data/processed/train"
val_dir = "../data/processed/val"

if not os.path.isdir(train_dir):
    print("train folder not found, run 01_data_prep.ipynb first")

In [4]:
# basic image settings for a first test run
image_size = 128
batch_size = 16
epochs = 15
learning_rate = 0.0001

# simple transform - resize and convert to tensor
transform = transforms.Compose([
    transforms.Resize((image_size, image_size)),
    transforms.ToTensor()
])

In [5]:
# load train and val data using folder names as class labels
train_data = datasets.ImageFolder(train_dir, transform=transform)
val_data = datasets.ImageFolder(val_dir, transform=transform)

# check the class order matches what we expect
print(train_data.class_to_idx)

train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_data, batch_size=batch_size, shuffle=False)

{'healthy': 0, 'low_tread': 1, 'sidewall_damaged': 2, 'uneven_wear': 3, 'zero_tread': 4}


In [8]:
# check class balance again - looking for one class dominating
counts = check_dataset('../data/processed/train')

total = sum(counts.values())
for class_name in counts:
    percent = counts[class_name] / total * 100
    print(class_name, ":", round(percent, 1), "%")

Dataset check:
healthy : 116 images 
low_tread : 35 images 
sidewall_damaged : 5 images   <- low, may need more images
uneven_wear : 14 images   <- low, may need more images
zero_tread : 26 images 
Total: 196 images
healthy : 59.2 %
low_tread : 17.9 %
sidewall_damaged : 2.6 %
uneven_wear : 7.1 %
zero_tread : 13.3 %


In [6]:
# build the model
num_classes = len(train_data.classes)
model = SimpleCNN(num_classes).to(device)

loss_function = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

In [ ]:
# training loop with early stopping
# Stop if BOTH train accuracy and validation accuracy don't change
# for 5 consecutive epochs.

patience = 5
unchanged_epochs = 0

previous_train_accuracy = None
previous_val_accuracy = None

for epoch in range(epochs):

    # =========================
    # TRAINING
    # =========================
    model.train()
    train_loss = 0
    correct = 0
    total = 0

    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)
        loss = loss_function(outputs, labels)

        loss.backward()
        optimizer.step()

        train_loss += loss.item()

        predicted = outputs.argmax(dim=1)
        correct += (predicted == labels).sum().item()
        total += labels.size(0)

    train_accuracy = correct / total


    # =========================
    # VALIDATION
    # =========================
    model.eval()
    val_correct = 0
    val_total = 0

    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)

            predicted = outputs.argmax(dim=1)

            val_correct += (predicted == labels).sum().item()
            val_total += labels.size(0)

    val_accuracy = val_correct / val_total


    # =========================
    # PRINT RESULTS
    # =========================
    print(
        "Epoch", epoch + 1,
        "- train loss:", round(train_loss, 3),
        "train acc:", round(train_accuracy, 3),
        "val acc:", round(val_accuracy, 3)
    )


    # =========================
    # EARLY STOPPING
    # =========================

    if previous_train_accuracy == train_accuracy and \
       previous_val_accuracy == val_accuracy:

        unchanged_epochs += 1

    else:
        unchanged_epochs = 0


    # Save current accuracies for next epoch
    previous_train_accuracy = train_accuracy
    previous_val_accuracy = val_accuracy


    # Stop if unchanged for 5 epochs
    if unchanged_epochs >= patience:
        print("Training stopped early!")
        print("Both train and validation accuracy did not change for",
              patience, "epochs.")
        break

Epoch 1 - train loss: 14.704 train acc: 0.592 val acc: 0.61
Epoch 2 - train loss: 14.904 train acc: 0.592 val acc: 0.61
Epoch 3 - train loss: 14.669 train acc: 0.592 val acc: 0.61
Epoch 4 - train loss: 14.796 train acc: 0.592 val acc: 0.61
Epoch 5 - train loss: 14.757 train acc: 0.592 val acc: 0.61
Epoch 6 - train loss: 14.921 train acc: 0.592 val acc: 0.61
Training stopped early!
Both train and validation accuracy did not change for 5 epochs.


In [9]:
# import the new metrics functions
import sys
sys.path.insert(0, '..')
from src.utils.metrics import get_predictions, show_confusion_matrix, show_classification_report

In [ ]:
# get predictions on val set
class_names = train_data.classes
true_labels, predicted_labels = get_predictions(model, val_loader, device)

show_confusion_matrix(true_labels, predicted_labels, class_names)

NameError: name 'model' is not defined

In [ ]:
# detailed per-class scores
show_classification_report(true_labels, predicted_labels, class_names)

In [ ]:
# save the trained model
os.makedirs("../results/cnn", exist_ok=True)
torch.save(model.state_dict(), "../results/cnn/model.pt")
print("Model saved to results/cnn/model.pt")